# Session 2 — Cleaning and Describing the Data

**Where we are.** In Session 1 we imported the raw file and listed five oddities. Today each oddity gets a decision and a reason. Then we describe the cleaned data and plot the first patterns.

Parts 1–4 are worked examples that follow the slides. Part 5 is the in-class exercise; it feeds your group presentation.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

raw = pd.read_excel("data/raw/default of credit card clients.xls", header=1)
df = raw.copy()          # we never modify `raw`
df.shape

## Part 1 — Cleaning is a sequence of decisions

### 1.1 Rename once, at the start
There is no `PAY_0`/`PAY_1` logic in the codebook: `PAY_0` is September, like `BILL_AMT1`. We rename it so that suffix 1 always means September.

In [ ]:
df = df.rename(columns={"PAY_0": "PAY_1",
                        "default payment next month": "default",
                        "LIMIT_BAL": "limit", "AGE": "age", "ID": "id"})

# six columns at a time: a dictionary built with a loop
df = df.rename(columns={f"PAY_{k}": f"pay_{k}" for k in range(1, 7)})
df = df.rename(columns={f"BILL_AMT{k}": f"bill_{k}" for k in range(1, 7)})
df = df.rename(columns={f"PAY_AMT{k}": f"paid_{k}" for k in range(1, 7)})
df.columns.tolist()

### 1.2 Missing values: explicit and disguised

In [ ]:
df.isna().sum().sum()     # no empty cells at all...

In [ ]:
# ...but the codebook only defines EDUCATION 1-4 and MARRIAGE 1-3
print(df["EDUCATION"].value_counts().sort_index(), "\n")
print(df["MARRIAGE"].value_counts().sort_index())

A **merge** with the codebook shows which codes it cannot match. The same tool combines two data sources.

In [ ]:
codebook = pd.DataFrame({
    "EDUCATION": [1, 2, 3, 4],
    "education": ["graduate school", "university", "high school", "others"]})

check = df.merge(codebook, on="EDUCATION", how="left", indicator=True)
check.groupby("_merge", observed=True)["EDUCATION"].unique()

**Decision (reference):** set the undocumented codes to missing. `.map()` does it for us: any code not in the dictionary becomes `NaN`.

In [ ]:
edu_labels = {1: "graduate school", 2: "university", 3: "high school", 4: "others"}
df["education"] = df["EDUCATION"].map(edu_labels)
df["marriage"] = df["MARRIAGE"].map({1: "married", 2: "single", 3: "others"})
df["sex"] = df["SEX"].map({1: "male", 2: "female"})

print(df[["education", "marriage", "sex"]].isna().sum())

### 1.3 Types: categories, with an order where it makes sense

In [ ]:
order = ["graduate school", "university", "high school", "others"]
df["education"] = pd.Categorical(df["education"], categories=order, ordered=True)
df["marriage"] = df["marriage"].astype("category")
df["sex"] = df["sex"].astype("category")
df = df.drop(columns=["EDUCATION", "MARRIAGE", "SEX"])
df.dtypes.head(8)

### 1.4 Duplicates

In [ ]:
dups = df[df.drop(columns="id").duplicated(keep=False)]
print(len(dups), "rows involved")
dups[["id", "limit", "age", "pay_1", "bill_1", "paid_1"]].head(6)

Different IDs, mostly inactive accounts with zero bills. **Decision:** keep them (two real clients can look identical).

### 1.5 Outliers: impossible or just unusual?

In [ ]:
print("negative September bills:", (df["bill_1"] < 0).sum())
print("largest limit:", df["limit"].max())
print("largest payment:", df["paid_1"].max())
print("age range:", df["age"].min(), "-", df["age"].max())

All of these are possible (a negative bill is a credit balance). **Decision:** keep them all.

## Part 2 — Creating variables

### Filtering

In [ ]:
young = df[df["age"] < 30]
risky = df[(df["pay_1"] >= 2) & (df["limit"] < 50_000)]
len(young), len(risky)

### Your Session 1 loop, for every client at once

In [ ]:
cols = [f"pay_{k}" for k in range(1, 7)]
late = df[cols] >= 1                  # True/False table
df["late_months"] = late.sum(axis=1)  # count the Trues across each row
df["late_months"].value_counts().sort_index()

### Utilisation and age groups

In [ ]:
df["utilisation"] = df["bill_1"] / df["limit"]
df["age_group"] = pd.cut(df["age"], bins=[20, 30, 40, 50, 80],
                         labels=["21-30", "31-40", "41-50", "51+"])
print("clients over their limit:", (df["utilisation"] > 1).sum())
df["age_group"].value_counts().sort_index()

### Save the clean file — Session 3 starts from here

In [ ]:
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/clients_clean.csv", index=False)

## Part 3 — Describing the data

In [ ]:
df[["limit", "age", "utilisation"]].describe().round(2)

In [ ]:
df["education"].value_counts(normalize=True, dropna=False).round(3)

In [ ]:
df.groupby("education", observed=True)["default"].agg(["mean", "size"]).round(3)

In [ ]:
pd.crosstab(df["education"], df["default"], normalize="index").round(3)

## Part 4 — Visualising

One question, one plot. Each title states what the plot shows.

In [ ]:
ax = sns.histplot(data=df, x="limit", bins=50)
ax.set(title="Credit limits are right-skewed", xlabel="Credit limit (NT$)")
plt.show()

In [ ]:
ax = sns.boxplot(data=df, x="limit", y="default", orient="h")
ax.set(title="Defaulters had lower credit limits", xlabel="Credit limit (NT$)",
       ylabel="Defaulted (1 = yes)")
plt.show()

In [ ]:
ax = sns.barplot(data=df, x="education", y="default")
ax.set(title="Default rate by education", xlabel="", ylabel="Default rate")
plt.show()

In [ ]:
ax = sns.barplot(data=df, x="late_months", y="default")
ax.set(title="Each extra late month raises the default rate",
       xlabel="Months paid late (out of 6)", ylabel="Default rate")
plt.show()

What are the black lines on top of each bar? Note your guess — Session 3 answers it.

---
## Part 5 — In-class exercise (feeds your group presentation)

Your group may make **different** cleaning decisions from the reference above. That is fine, as long as each is justified.

### Task 1 — Cleaning log
Complete the table. Add one row per decision (at least five rows).

| Issue | Decision | Reason | Rows affected |
|---|---|---|---|
| | | | |

### Task 2 — Check your cleaned data
Show that the cleaning did what the log says: count the missing values you created and check the dtypes.

In [ ]:
# TODO: your code here


### Task 3 — Describe the variables of one research question
Choose RQ1, RQ2 or RQ3. Produce summary statistics and one `groupby`.

In [ ]:
# TODO: your code here


### Task 4 — One plot that answers your question
The title must state what the plot shows. Then write two sentences: what the plot shows, and one reason to be careful with it.

In [ ]:
# TODO: your code here


*Your two sentences:*